# Eksperiment: utjecaj perturbacije 5 tezina u zadanom layeru

Cilj: pronaci 5 tezina unutar danog layera, kada se promijene
malim Gaussovim šumom, uzrokuju najvecu degradaciju performanse modela.

Pristup:
1. Ucitavam *Qwen2.5-0.5B-Instruct* i smrzavam sve parametre.
2. Definiramo evaluacijsku metriku (prosjecni cross-entropy loss na fiksnom skupu recenica).
3. Iterativni algoritam pretrage:
   - radi samo na `model.model.layers[-1].self_attn` (u trenutnim postavkama)
   - svaka iteracija: odabir 5 tezina -> Gaussov sum -> eval -> trenutna obnova
   - prati najgoru (najvecu) vrijednost loss-a i koje su je tezine uzrokovale
4. Ako stagnira, mijenja strategiju odabira tezina (ne pojačava sum):
   - random -> magnitude (top-N po apsolutnoj vrijednosti) -> uze magnitude pool -> random

In [1]:
# === Imports i seed ===
import torch
import numpy as np
import random
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


In [2]:
# UCITAVANJE MODELA I TOKENIZERA
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # ucitava tokenizer za zadani model, to je kao prevoditelj iz recenice u vektor indeksa tokena u rijecniku
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32) # ovo ucitava model
model.to(device)
model.eval() # prebaci model u mod za predictanje (drugi mod je mod za ucenje, to ne zelim)

# smrznemo sve parametre, nikad ne zelimo gradijente niti slučajne update-ove
for p in model.parameters():
    p.requires_grad = False # ovo je isto ugl. za treniranje

print(f"Ucitan model: {MODEL_NAME}")
print(f"Ukupno parametara: {sum(p.numel() for p in model.parameters()):,}")


Device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Ucitan model: Qwen/Qwen2.5-0.5B-Instruct
Ukupno parametara: 494,032,768


In [3]:
# === Evaluacija ===
QA_PAIRS = [
    ("Calculate 2 + 2. Reply with the result number only.\nAnswer:", " 4"),
    ("Calculate 5 + 3. Reply with the result number only.\nAnswer:", " 8"),
    ("Calculate 10 - 7. Reply with the result number only.\nAnswer:", " 3"),
    ("Calculate 7 * 6. Reply with the result number only.\nAnswer:", " 42"),
    ("Calculate 100 / 4. Reply with the result number only.\nAnswer:", " 25"),
    ("What is the capital of France? Reply with one word.\nAnswer:", " Paris"),
    ("What color is the sky on a clear day? Reply with one word.\nAnswer:", " blue"),
    ("What is the largest planet in our solar system? Reply with one word.\nAnswer:", " Jupiter"),
] # lista parova, tj. prompt + ocekivani odgovor

@torch.no_grad() # iskljuci pracenje gradijenta za sve operacije unutar funkcije
def evaluate_model(model, tokenizer, qa_pairs, device):
    """
    Za svaki (prompt, answer) par:
      1) tokeniziramo prompt i answer odvojeno
      2) forward pass kroz model -> dobivamo logite za svaku poziciju
      3) izvlačimo logite koji predvidjaju ANSWER tokene
      4) softmax -> vjerojatnosti
      5) iz vjerojatnosti uzimamo onu koju je model dao TOCNOM answer tokenu
      6) log loss = -log(P_tocan) po tokenu
      7) prosjek po answer tokenima = loss za par
    Finalni rezultat = prosjek po svim parovima.
    """
    pair_losses = [] # ovdje skupljam lossove

    for prompt, answer in qa_pairs: # iteriram se po parovima
        # TOKENIZACIJA
        prompt_ids = tokenizer(prompt, return_tensors="pt",
                               add_special_tokens=False).input_ids[0] # dobivam tokene za pitanje (input_ids[0] jer je samo jedno pitanje u batchu)
        answer_ids = tokenizer(answer, return_tensors="pt",
                               add_special_tokens=False).input_ids[0] # dobivam tokene za odgovor
        prompt_len = len(prompt_ids)
        answer_len = len(answer_ids)

        # FORWARD PASS KROZ MODEL
        full_ids = torch.cat([prompt_ids, answer_ids]).unsqueeze(0).to(device) # spajam prompt i answer, unsqueeze pretvara u batch jer model ocekuje batch
        out = model(input_ids=full_ids)
        logits = out.logits # ovdje se dobije nešto u ovome stilu (batch_primjer, token, vokabular)
                            # batch_primjer se odnosi na recenicu, tj. dio batcha (ali ja imam samo jednu recenicu u batchu pa je ovo nevazna dimenzija)
                            # token se odnosi na neki token u mojoj recenici. za svaki token se uzima logits vektor vokabulara

        # IZVLACIM LOGITE KOJI PREDVIDAJU ANSWER TOKENE
        # Logit na poziciji t predvidjaa token na poziciji t+1.
        # Pa da predvidim answer_ids[0], trebam logits[0, prompt_len - 1].
        # Da predvidim answer_ids[i], trebam logits[0, prompt_len - 1 + i].
        start = prompt_len - 1
        end = start + answer_len
        answer_logits = logits[0, start:end] # (answer_len, vocab_size), dakle ako ocekujemo jedan token nakon answera, onda cemo dobiti
                                             # nesto oblika (1, vocab_size), tj. za svaki moguci token dobimo logit koji se moze pretvorit
                                             # u vjerojatnost

        # SOFTMAX
        probs = torch.softmax(answer_logits, dim=-1) # (answer_len, vocab_size), pretvara logite u vjerojatnosti (dobivam distribuciju)

        # IZVLACIM P(točan token) ZA SVAKI ANSWER TOKEN
        # probs[i, answer_ids[i]] = vjerojatnost koju je model dao i-tom answer tokenu
        target_probs = probs[torch.arange(answer_len), answer_ids.to(device)] # torch.arange(answer_len) bira redove koje zelim (moze ih biti vise ako imamo vise answer tokena)
                                                                              # answer_ids.to(device) bira stupce koje zelim (svaki red je jedan answer token, a jedan red moze
                                                                              # potencijalno poprimit vrijednost bilo kojeg tokena iz vokabulara, a mi gledamo vjerojatnost
                                                                              # samo za ono sto nas zanima). Npr.:
                                                                              # probs[0, 220] = P(token=220 nakon prompta)             = npr. 0.85
                                                                              # probs[1, 19]  = P(token=19 nakon prompta+token_220)    = npr. 0.92


        # LOG LOSS PO TOKENU
        token_losses = -torch.log(target_probs) # (answer_len,)

        # AKO IMA VISE ANSWER TOKENA, RACUNAM PROSJEK
        pair_loss = token_losses.mean().item()
        pair_losses.append(pair_loss)

    return float(np.mean(pair_losses))

In [4]:
# === Ciljani layer i inventar težina ===
def get_target_layer(model):
    """Zadnji self-attention modul (q_proj, k_proj, v_proj, o_proj zajedno)."""
    return model.model.layers[-1].self_attn # uzmem zadnji transformer blok i od tamo uzmem attention layer

def get_weight_inventory(layer):
    """
    Vraca listu (name, parameter, numel) za sve 'weight' matrice u layeru.
    Bias-evi se preskacu (nisu uvijek prisutni i nisu fokus eksperimenta).
    """
    inv = []
    for name, p in layer.named_parameters(): # named_parameters vraca iterator parova (ime, parametar), npr. ("q_proj.weight", tensor(896×896))
        if "weight" in name: # ignoriramo biase
            inv.append((name, p, p.numel())) # p.numel() vraca ukupan broj elemenata u tenzoru
    return inv

target_layer = get_target_layer(model)
inventory = get_weight_inventory(target_layer)

print("Ciljani layer:", type(target_layer).__name__)
print("Tezine u prostoru pretrage:")
for name, p, n in inventory:
    print(f"  {name}: shape={tuple(p.shape)}, numel={n:,}")
print(f"Ukupno tezina: {sum(n for _,_,n in inventory):,}")


Ciljani layer: Qwen2Attention
Tezine u prostoru pretrage:
  q_proj.weight: shape=(896, 896), numel=802,816
  k_proj.weight: shape=(128, 896), numel=114,688
  v_proj.weight: shape=(128, 896), numel=114,688
  o_proj.weight: shape=(896, 896), numel=802,816
Ukupno tezina: 1,835,008


In [5]:
# === Strategije odabira težina ===
def _global_to_local(global_idx, inventory):
    """Globalni flat index -> (param_name, lokalni flat index unutar te matrice)."""
    # neka generalna ideja ove funkcije je da ide kroz inventory redom i akumulira koliko smo
    # do sada prosli. Dakle ako kao parametar predam odredjeni globalni indeks, dobit cu gdje se lokalno
    # nalazi taj weight
    s = 0
    for name, _, n in inventory:
        if global_idx < s + n:
            return name, global_idx - s # vracam (name, i indeks u tom parametru)
        s += n
    raise IndexError(global_idx)

def random_selection(inventory, k=5, rng=None):
    """STRATEGIJA 1: Uniformno nasumicnih k težina iz cijelog layera."""
    if rng is None:
        rng = random # odabir random generatora
    total = sum(n for _, _, n in inventory) # racuna ukupan broj tezina
    chosen = rng.sample(range(total), k) # bira k razlicitih elemenata iz populacije BEZ ponavljanja
    return [_global_to_local(g, inventory) for g in chosen] # prevodi svaki globalni indeks u par (ime, lokalni_indeks)

def magnitude_selection(inventory, k=5, top_pool=2000, rng=None):
    """
    STRATEGIJA 2: bira k težina iz skupa top-`top_pool` po apsolutnoj vrijednosti.
    Hipoteza: tezine s velikom magnitudom imaju veći utjecaj kad se naruse.
    """
    if rng is None:
        rng = random
    abs_vals = torch.cat([p.detach().abs().view(-1) for _, p, _ in inventory]) # spajam sve tezine u jedan dugi vektor
          # koristim abs jer me samo zanima magnituda tezine. view(-1) pretvara matricu u jednodimenzionalni vektor
          # dakle iz (896, 896) u (802816,). for petlja radi sve navedeno za svaku tezinu u inventoryju, rezultat
          # je lista od 4 tenzora. cat spaja ta 4 tenzora u jedan veliki tenzor
    pool = min(top_pool, abs_vals.numel()) # pazimo da netko ne odabere veličinu poola vecu od onoga sto trenutno imamo
    _, topk_idx = torch.topk(abs_vals, pool) # pronadji top k najboljih, vrati dva tenzora, vrijednosti i indekse u orginalnom tenzoru (trebamo samo indekse) (globalni indeksi)
    chosen_in_pool = rng.sample(range(pool), k) # biramo odredjeni broj najboljih
    chosen_global = [int(topk_idx[c].item()) for c in chosen_in_pool] # pretvaram pool pozicije u globalne indekse
    return [_global_to_local(g, inventory) for g in chosen_global] # za kraj pretvaram u (ime, indeks) parove


In [6]:
# === Perturbacija i obnova ===
def apply_perturbation(layer, selections, std=0.5):
    """
    Na svaku odabranu tezinu dodaje N(0, std). Vraca backup originalnih vrijednosti
    kako bismo ih mogli vratiti.
    """
    # layer - modul u kojem mijenjamo tezine
    # selections - parovi (ime_parametra, lokalni_indeks)
    # std - standardna devijacija
    backups = [] # pamtimo sto je bila orginalna tezina prije svake izmjene
    params = dict(layer.named_parameters()) # layer.named_parameters() vraca iterator parova (ime, parametar) koji se pretvara u dict za laksi lookup
    for name, idx in selections: # iteriram se kroz sve odabrane tezine, svaku odaberemo nezavisno
        p = params[name] # pristupam parametru
        flat = p.data.view(-1) # uzima sirovi tenzor (pomocu data) te pretvara ga u 1D array za laksi pristup
        orig = flat[idx].item() # prije ikakvih izmjena, spremam trenutnu vrijednost
        noise = float(np.random.normal(0.0, std)) # generiram Gaussov sum
        new_val = orig + noise # uzimam orginalnu vrijednost i dodam Gaussov sum
        flat[idx] = new_val # izmjenim tezinu
        backups.append((name, idx, orig, new_val)) # pamtim ime, lokalni_indeks i originalnu vrijednost u backup listi za kasnije
    return backups

def restore_weights(layer, backups):
    """Vraca sve perturbirane tezine na originalne vrijednosti (in-place)."""
    # ovo je inverzna funkcija prethodnoj funkciji
    params = dict(layer.named_parameters()) # isto kao i u prethodnoj funkciji
    for entry in backups:
        name, idx, orig = entry[0], entry[1], entry[2]
        params[name].data.view(-1)[idx] = orig
        # params[name] - parametar tog imena
        # data - sirovi tenzor
        # view(-1) - raw view tog tenzora


In [7]:
# === Glavni algoritam pretrage ===
def search_worst_perturbation(
    model, tokenizer, sentences, device,
    n_iterations=200,
    k=5,
    noise_std=0.5,
    patience=30,
    initial_top_pool=1000,
    seed=123,
):
    """
    Iterativna pretraga 5 tezina koje uzrokuju najvecu degradaciju.

    - n_iterations : ukupan broj pokusaja
    - k            : broj tezina po iteraciji (zadatak trazi tocno 5)
    - noise_std    : std Gaussovog suma
    - patience     : nakon koliko iteracija bez napretka mijenjamo strategiju
    """
    layer = get_target_layer(model) # uzima layer
    inv = get_weight_inventory(layer) # na temelju layera radi inventory kao listu jedinki (name, p, p.numel()), za svaki nameani parametar (tj. grupa weightova)

    baseline_loss = evaluate_model(model, tokenizer, sentences, device) # mjerim jednom performansu baseline modela (referentna tocka)
    print(f"Baseline loss: {baseline_loss:.6f}\n")

    # stanje algoritma
    worst_loss = baseline_loss
    worst_selections = None
    worst_originals = None
    iters_since_improve = 0
    strategy = "random"
    top_pool = initial_top_pool
    rng = random.Random(seed)

    for it in range(1, n_iterations + 1):
        # SELEKCIJA TEZINA
        # biram k tezina
        if strategy == "random":
            selections = random_selection(inv, k=k, rng=rng)
        else:
            selections = magnitude_selection(inv, k=k, top_pool=top_pool, rng=rng)

        # PERTUBACIJA -> EVAL -> OBNOVA
        backups = apply_perturbation(layer, selections, std=noise_std) # pamti orginalnu vrijednost i izmjeni trenutnu za k tezina
        loss = evaluate_model(model, tokenizer, sentences, device) # mjeri loss na perturbiranim modelom
        restore_weights(layer, backups) # vraca tezine na orginalne vrijednosti

        # PRACENJE NAJGOREG SLUCAJA
        if loss > worst_loss:
            worst_loss = loss # novi najgori gubitak
            worst_selections = list(selections) # kopija liste, selections se rekreira u svakoj iteraciji
            worst_originals = list(backups) # -||-
            iters_since_improve = 0
            print(f"[{it:3d}] strat={strategy:9s} loss={loss:.6f}  "
                  f"(delta={loss-baseline_loss:+.4f}) => novi maksimum")
        else:
            iters_since_improve += 1 # ako nije nadjeno bolje, iteriramo counter stagnacije

        # PROMJENA STRATEGIJE AKO STAGNIRAMO
        if iters_since_improve >= patience:
            if strategy == "random":
                strategy = "magnitude" # promjena strategije
                iters_since_improve = 0
                print(f"\n--- Stagnacija {patience} iter: prelazimo na MAGNITUDE "
                      f"(top {top_pool}) ---\n")
            else: # ako smo u magnitudskoj strategiji, a i dalje stagniramo, suzvamo pool
                new_pool = max(50, top_pool // 2)
                if new_pool != top_pool:
                    top_pool = new_pool
                    iters_since_improve = 0
                    print(f"\n--- Stagnacija: suzavamo magnitude pool na top {top_pool} ---\n")
                else:
                    strategy = "random"
                    iters_since_improve = 0
                    print(f"\n--- Stagnacija: vracamo se na RANDOM radi diverziteta ---\n")

    return baseline_loss, worst_loss, worst_selections, worst_originals


In [8]:
# POKRECEM PRETRAGU
baseline, worst, sel, originals = search_worst_perturbation(
    model, tokenizer, QA_PAIRS, device,
    n_iterations=200,
    k=5,
    noise_std=0.5,
    patience=30,
)


Baseline loss: 0.109789

[  1] strat=random    loss=0.110204  (delta=+0.0004) => novi maksimum
[  2] strat=random    loss=0.110247  (delta=+0.0005) => novi maksimum
[  4] strat=random    loss=0.111128  (delta=+0.0013) => novi maksimum
[ 10] strat=random    loss=0.111256  (delta=+0.0015) => novi maksimum
[ 16] strat=random    loss=0.111582  (delta=+0.0018) => novi maksimum
[ 21] strat=random    loss=0.117543  (delta=+0.0078) => novi maksimum

--- Stagnacija 30 iter: prelazimo na MAGNITUDE (top 1000) ---


--- Stagnacija: suzavamo magnitude pool na top 500 ---

[ 84] strat=magnitude loss=0.136266  (delta=+0.0265) => novi maksimum

--- Stagnacija: suzavamo magnitude pool na top 250 ---


--- Stagnacija: suzavamo magnitude pool na top 125 ---


--- Stagnacija: suzavamo magnitude pool na top 62 ---



In [9]:
# ZAVRSNI IZVJESTAJ
print("=" * 72)
print("REZULTAT")
print("=" * 72)
print(f"Pocetni loss          : {baseline:.6f}") # exp(loss) = exp(-log(P)) = 1/P
print(f"Najgori loss          : {worst:.6f}")
print(f"Apsolutna degradacija : {worst - baseline:+.6f}")
print(f"Relativna degradacija : {(worst - baseline) / baseline * 100:+.2f}%")
print()
print("5 tezina koje su uzrokovale najvecu degradaciju:")
print("-" * 72)

layer = get_target_layer(model) # dohvacam layer
params = dict(layer.named_parameters()) # layer.named_parameters() vraca iterator parova (ime, parametar) koji se pretvara u dict za laksi lookup

if originals is not None:
    for i, (name, idx, orig, new_val) in enumerate(originals, 1): # originals sadrzi podataka i o IZMJENJENIM TEZINIMA (iako se koristi za restoreanje, pogledat gore)
        shape = tuple(params[name].shape)
        multi = np.unravel_index(idx, shape) # (row, col) prikaz
        noise = new_val - orig  # rekonstrukcija šuma iz orig i new_val
        print(f"  {i}. param='{name}'  flat_idx={idx:>7d}  shape_idx={multi}")
        print(f"     original={orig:+.6f}  ->  perturbed={new_val:+.6f}  (noise={noise:+.6f})")
else:
    print("  Nije pronadjena perturbacija koja pogorsava model vise od baseline-a.")


REZULTAT
Pocetni loss          : 0.109789
Najgori loss          : 0.136266
Apsolutna degradacija : +0.026477
Relativna degradacija : +24.12%

5 tezina koje su uzrokovale najvecu degradaciju:
------------------------------------------------------------------------
  1. param='o_proj.weight'  flat_idx= 774529  shape_idx=(np.int64(864), np.int64(385))
     original=-0.157227  ->  perturbed=-0.531470  (noise=-0.374243)
  2. param='o_proj.weight'  flat_idx= 355233  shape_idx=(np.int64(396), np.int64(417))
     original=-0.208008  ->  perturbed=+0.567568  (noise=+0.775576)
  3. param='o_proj.weight'  flat_idx= 745018  shape_idx=(np.int64(831), np.int64(442))
     original=-0.196289  ->  perturbed=-0.138452  (noise=+0.057837)
  4. param='v_proj.weight'  flat_idx=  38454  shape_idx=(np.int64(42), np.int64(822))
     original=+0.156250  ->  perturbed=+0.745899  (noise=+0.589649)
  5. param='o_proj.weight'  flat_idx= 439130  shape_idx=(np.int64(490), np.int64(90))
     original=-0.468750  ->  pe

In [10]:
# VERIFIKACIJA JE LI MODEL NETAKNUT
final_loss = evaluate_model(model, tokenizer, QA_PAIRS, device)
diff = abs(final_loss - baseline)

print(f"Trenutni loss modela : {final_loss:.6f}")
print(f"Pocetni (baseline)   : {baseline:.6f}")
print(f"Razlika              : {diff:.2e}")
print("Model je u potpunosti netaknut." if diff < 1e-6
      else "POZOR: model je trajno modificiran!")


Trenutni loss modela : 0.109789
Pocetni (baseline)   : 0.109789
Razlika              : 0.00e+00
Model je u potpunosti netaknut.


(Koristen je Claude by Anthropic za pisanje koda.)